# Code Review Assistant: eval harness

Companion notebook to the [Gradio Space](https://huggingface.co/spaces). Runs the canonical v3 prompt against a hand-curated eval set and documents the v1, v2, v3 iteration that produced it.

**What this notebook proves**

1. The prompt is schema-locked via forced tool use (OpenAI-compatible API on the HF router).
2. Iteration is eval-driven, not vibes-driven. The fabricated-`major`-on-clean-code count goes 2, then 3, then 0 across v1 to v3. v2 is a regression worth naming; v3 is the win.
3. Pass rate alone does not capture the win. All three versions land at 6/8 pass. The failure *profile* shifts: v1/v2 fail on the safety-critical axis (manufacturing bugs on clean code); v3 fails on under-detection of structural complexity, the recoverable failure mode for a deployed reviewer.

**To run**: set your `HF_TOKEN` via Colab Secrets (key icon in the sidebar). Generate a fine-grained token with the *Make calls to Inference Providers* permission at https://huggingface.co/settings/tokens.

## 1. Install + imports

In [ ]:
%pip install -q openai jsonschema

In [ ]:
import json, os, re, time, urllib.request
from dataclasses import dataclass, field
from typing import Any

from jsonschema import ValidationError, validate
from openai import OpenAI

try:
    from google.colab import userdata  # type: ignore
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    assert 'HF_TOKEN' in os.environ, 'Set HF_TOKEN in Colab Secrets or your shell.'

# If opened in Colab (or anywhere without the sibling files), the loader below
# falls back to fetching from the HF Space's raw URL.
SPACE_OWNER = os.environ.get('SPACE_OWNER', 'faya98')
SPACE_NAME = os.environ.get('SPACE_NAME', 'code-reviewer')
RAW_BASE = f'https://huggingface.co/spaces/{SPACE_OWNER}/{SPACE_NAME}/raw/main'

def load_text(local_path: str, remote_path: str) -> str:
    if os.path.exists(local_path):
        return open(local_path, encoding='utf-8').read()
    url = f'{RAW_BASE}/{remote_path}'
    with urllib.request.urlopen(url) as resp:
        return resp.read().decode('utf-8')

MODEL_ID = 'Qwen/Qwen3-Coder-30B-A3B-Instruct:fastest'
client = OpenAI(base_url='https://router.huggingface.co/v1', api_key=os.environ['HF_TOKEN'])
print('Model:', MODEL_ID)

## 2. Load eval set and tool schema

The eval set lives next to this notebook at `snippets.json` (also acts as the public dataset reference). Eight hand-curated snippets across Python + TypeScript, four categories: `buggy`, `subtle`, `well-written`, `edge`.

In [ ]:
snippets = json.loads(load_text('snippets.json', 'eval/snippets.json'))

print(f'{len(snippets)} snippets:')
for s in snippets:
    print(f"  {s['id']:<30} {s['language']:<11} {s['category']}")

In [ ]:
REVIEW_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'required': ['positive_note', 'improvements', 'overall_assessment'],
    'properties': {
        'positive_note': {'type': 'string', 'minLength': 1},
        'improvements': {
            'type': 'array', 'minItems': 3, 'maxItems': 3,
            'items': {
                'type': 'object', 'additionalProperties': False,
                'required': ['dimension', 'severity', 'issue', 'why_it_matters', 'suggested_fix'],
                'properties': {
                    'dimension': {'type': 'string', 'enum': ['readability', 'structure', 'maintainability']},
                    'severity': {'type': 'string', 'enum': ['major', 'minor', 'nit']},
                    'issue': {'type': 'string', 'minLength': 1},
                    'why_it_matters': {'type': 'string', 'minLength': 1},
                    'suggested_fix': {'type': 'string', 'minLength': 1},
                },
            },
        },
        'overall_assessment': {'type': 'string', 'enum': ['needs_work', 'solid', 'exemplary']},
    },
}

SUBMIT_REVIEW_TOOL = {
    'type': 'function',
    'function': {
        'name': 'submit_review',
        'description': 'Submit a structured code review. Always call this tool.',
        'parameters': REVIEW_SCHEMA,
    },
}

## 3. The three prompt versions

Each version is shipped against the same eval set. The metric that matters is the **delta**: where did the next iteration fix something measurable?

In [ ]:
PROMPT_V1 = '''You are a code reviewer. Review the user's code snippet and suggest three improvements. Also say one good thing about the code.'''

PROMPT_V2 = '''You are a senior code reviewer. Review the user's snippet across three dimensions: readability, structure, maintainability.

Call the `submit_review` tool with:
- positive_note: one sentence on what the code does well
- improvements: exactly three items, each tagged with dimension (readability|structure|maintainability), an issue, why it matters, and a suggested fix
- overall_assessment: needs_work | solid | exemplary'''

# Canonical, shipped prompt. Same file the Gradio app loads.
PROMPT_V3 = load_text('../prompts/system_prompt.md', 'prompts/system_prompt.md')

STRICT_RETRY_SUFFIX = (
    '\n\nIMPORTANT: Your previous response failed schema validation. '
    'Return exactly one call to the `submit_review` tool with exactly three '
    'items in improvements. Do not include any other content.'
)

## 4. Runner + three-layer defensive parser

Mirrors the deployed app's behaviour exactly. Each call goes:
1. Forced tool call → schema validate.
2. On failure, one retry with a stricter prompt suffix.
3. On failure, regex-extract a JSON block from any text.

We track which layer landed each result; the **fallback rate** (anything not in layer 1) is the secondary quality metric per prompt version.

In [ ]:
JSON_BLOCK_RE = re.compile(r'\{[\s\S]*\}')

def _call(system, code, language):
    return client.chat.completions.create(
        model=MODEL_ID, max_tokens=1024,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': f'Language: {language}\n\nSnippet:\n```{language}\n{code}\n```\n\nReview by calling the `submit_review` tool.'},
        ],
        tools=[SUBMIT_REVIEW_TOOL],
        tool_choice={'type': 'function', 'function': {'name': 'submit_review'}},
    )

def _extract_tool(response):
    try:
        tc = response.choices[0].message.tool_calls or []
        return json.loads(tc[0].function.arguments) if tc else None
    except Exception:
        return None

def _extract_text(response):
    try:
        text = response.choices[0].message.content or ''
    except Exception:
        return None
    m = JSON_BLOCK_RE.search(text)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def _valid(payload):
    if not payload:
        return False
    try:
        validate(payload, REVIEW_SCHEMA); return True
    except ValidationError:
        return False

def review(system, code, language):
    '''Returns (payload, layer) where layer ∈ {ok, retry, regex, failed}.'''
    resp = _call(system, code, language)
    p = _extract_tool(resp)
    if _valid(p): return p, 'ok'
    resp = _call(system + STRICT_RETRY_SUFFIX, code, language)
    p = _extract_tool(resp)
    if _valid(p): return p, 'retry'
    p = _extract_text(resp)
    if _valid(p): return p, 'regex'
    return None, 'failed'

## 5. The judge

Four checks per snippet (per `rubric.md`):

1. Schema valid (already handled by the runner; we just record the layer).
2. `expected_dimensions ⊆` returned dimensions.
3. `overall_assessment` matches `expected_assessment` (with `solid ↔ exemplary` flex on well-written).
4. **Headline check**: on `category: well-written`, no improvement has `severity: major`.

Every improvement's `issue` text is printed inline so manual scan-for-fabrication takes one scroll.

In [ ]:
@dataclass
class Verdict:
    snippet_id: str
    layer: str
    passed: bool
    notes: list[str] = field(default_factory=list)
    payload: dict | None = None

def judge(snippet, payload, layer) -> Verdict:
    notes = []
    passed = True
    if layer == 'failed' or payload is None:
        return Verdict(snippet['id'], layer, False, ['all three response layers failed'])
    expected_dims = set(snippet.get('expected_dimensions', []))
    actual_dims = {imp['dimension'] for imp in payload['improvements']}
    if expected_dims and not expected_dims.issubset(actual_dims):
        passed = False
        notes.append(f'missing dimensions: {expected_dims - actual_dims}')
    expected_a = snippet['expected_assessment']
    actual_a = payload['overall_assessment']
    flex_pair = {expected_a, actual_a} == {'solid', 'exemplary'}
    if actual_a != expected_a and not flex_pair:
        passed = False
        notes.append(f'assessment {actual_a!r} ≠ expected {expected_a!r}')
    if snippet['category'] == 'well-written':
        majors = [i for i in payload['improvements'] if i['severity'] == 'major']
        if majors:
            passed = False
            notes.append(f'HONESTY FAIL: {len(majors)} major item(s) on well-written code')
    return Verdict(snippet['id'], layer, passed, notes, payload)

def run_version(name, system_prompt):
    print(f'\n=== {name} ===')
    verdicts = []
    for s in snippets:
        payload, layer = review(system_prompt, s['code'], s['language'])
        v = judge(s, payload, layer)
        verdicts.append(v)
        flag = '✓' if v.passed else '✗'
        suffix = '' if v.layer == 'ok' else f' [{v.layer}]'
        print(f"  {flag} {s['id']:<30} {s['category']:<13}{suffix}")
        for n in v.notes:
            print(f'      → {n}')
        if v.payload:
            for imp in v.payload['improvements']:
                print(f"      ▸ [{imp['severity']:<5} {imp['dimension']:<15}] {imp['issue']}")
    pass_n = sum(1 for v in verdicts if v.passed)
    fb_n = sum(1 for v in verdicts if v.layer != 'ok')
    print(f'\n  pass rate:     {pass_n}/{len(verdicts)} ({pass_n/len(verdicts)*100:.0f}%)')
    print(f'  fallback rate: {fb_n}/{len(verdicts)} ({fb_n/len(verdicts)*100:.0f}%)')
    return verdicts

### v1: naive prompt

*"You are a code reviewer. Review the user's code snippet and suggest three improvements."*

Hypothesis: no tool schema, no rubric dimensions, no honesty guard. Will produce well-meaning prose that occasionally happens to be JSON-shaped, and will manufacture issues on clean code because the prompt demands three improvements.

**Expected failure modes**: high fallback-parser rate (because no `tool_choice` forces JSON); major-severity items on well-written snippets (because no honesty rule).

In [ ]:
v1_results = run_version('v1: naive prompt', PROMPT_V1)

### v2: add rubric + forced tool call

Add the three named dimensions (readability, structure, maintainability). Force `tool_choice` on `submit_review`. This kills the JSON-shape problem.

**Expected delta**: fallback rate drops to near zero. But the honesty issue should remain: the prompt still says "return three improvements" without permission to mark them as nits, so the model will invent defects on clean code.

In [ ]:
v2_results = run_version('v2: rubric + forced tool call', PROMPT_V2)

### v3: the honesty rule + severity field

Introduce the `severity: major | minor | nit` field. Define `nit` as the honest landing zone for polish-level suggestions on clean code. Add explicit "do not invent defects" instruction. This resolves the literal-rubric vs honest-output trade-off: the model still returns three items, but they degrade to nits when the code is genuinely clean.

**Expected delta**: fabricated-major count on clean code drops to zero. **Trade-off to watch**: a stricter definition of `major` may also under-flag complexity-class issues on subtle and edge snippets.

> **Iteration notes (real, not staged)**
>
> 1. An early build of v3 still flagged a *speculative* concern (missing regex-group-count validation) as `severity: major` on the `parse_iso_duration` snippet during smoke testing. The fix was a worked counter-example in the system prompt: `major` is reserved for defects in the *current* snippet (bugs, security holes, type-safety leaks); hypotheticals about future changes can be at most `minor`, and on otherwise clean code, `nit`. The shipped prompt is the post-tightening version.
>
> 2. **The eval found a bug in the eval.** The original `ts-well-written-1` snippet used an unchecked `(await res.json()) as T` cast. The model flagged that as `major`, correctly per the system prompt's own definition of `major` ("unchecked `as T` assertion on `any[]` input"). The snippet was not actually well-written. The dataset has been fixed (replaced with a debounce helper that has no type assertions).
>
> 3. **v3's residual failure mode is calibration on complexity, not honesty.** Below, v3 hits **0 fabricated majors** on the well-written set, but under-rates `ts-subtle-deep-nesting` and `ts-edge-any-leak` as `solid` instead of `needs_work`. The "What major actually means" section in the system prompt is bug-and-security-focused; complexity-as-defect needs its own worked example to bring v3 to 8/8. That is a one-prompt-iteration fix away, with the eval as the regression suite that proves it.

In [ ]:
v3_results = run_version('v3: severity + honesty rule (canonical, shipped)', PROMPT_V3)

## 7. Side-by-side summary

In [ ]:
def summarise(name, results):
    pass_n = sum(1 for v in results if v.passed)
    fb_n = sum(1 for v in results if v.layer != 'ok')
    major_on_clean = 0
    subtle_pass = 0
    subtle_total = 0
    sev_totals = {'major': 0, 'minor': 0, 'nit': 0}
    for v, s in zip(results, snippets):
        if v.payload:
            for imp in v.payload['improvements']:
                sev_totals[imp['severity']] += 1
            if s['category'] == 'well-written':
                major_on_clean += sum(1 for i in v.payload['improvements'] if i['severity'] == 'major')
            if s['category'] == 'subtle':
                subtle_total += 1
                has_major = any(i['severity'] == 'major' for i in v.payload['improvements'])
                if has_major and v.payload['overall_assessment'] == 'needs_work':
                    subtle_pass += 1
    n = len(results)
    return {
        'version': name,
        'pass_rate': f'{pass_n}/{n} ({pass_n/n*100:.0f}%)',
        'fallback_rate': f'{fb_n}/{n} ({fb_n/n*100:.0f}%)',
        'major_on_well_written': major_on_clean,
        'subtle_detect': f'{subtle_pass}/{subtle_total}',
        'severity_dist': sev_totals,
    }

rows = [summarise('v1 naive', v1_results), summarise('v2 rubric+tool', v2_results), summarise('v3 honesty+severity', v3_results)]
for r in rows:
    print(r)

## 8. Per-snippet verdict table (v3)

What the shipped prompt does on each input.

In [ ]:
print(f"{'snippet':<30} {'category':<13} {'verdict':<8} {'sev breakdown':<22} {'assess':<11} {'layer'}")
print('-' * 100)
for v, s in zip(v3_results, snippets):
    if v.payload:
        sev = {'major': 0, 'minor': 0, 'nit': 0}
        for i in v.payload['improvements']:
            sev[i['severity']] += 1
        sev_str = f"M{sev['major']} m{sev['minor']} n{sev['nit']}"
        assess = v.payload['overall_assessment']
    else:
        sev_str = '-'
        assess = '-'
    flag = 'PASS' if v.passed else 'FAIL'
    print(f"{s['id']:<30} {s['category']:<13} {flag:<8} {sev_str:<22} {assess:<11} {v.layer}")

## What this proves

- Each iteration fixed a *measurable* failure on a fixed eval set, not a vibes rewrite. The fabricated-major-on-clean-code count goes **2, then 3, then 0** across v1, v2, v3. v2 is a regression worth naming.
- **Pass rate alone does not capture the win.** All three versions land at 6/8 pass; the failure *profile* shifts. v1/v2 fail on the safety-critical axis (manufacturing bugs on clean code). v3 fails on under-detection of structural complexity in TypeScript snippets, which is the recoverable failure mode for a deployed reviewer.
- **v2 surprises.** Adding rubric plus forced tool call made honesty *slightly worse* (2 to 3 fabricated majors). **Structured output is not the same as calibrated output.** Without permission to mark items as polish, the model fills the rubric by manufacturing defects.
- Output reliability is multi-layered: forced tool call first, prompt retry second, raw-text JSON extraction third. The fallback-parser rate is itself a tracked metric (0% across all three versions on this eval set).
- **The eval found a bug in itself.** The original `ts-well-written-1` snippet used an unchecked `as T` cast; the model correctly flagged it as `major` per the prompt's own definition. The snippet was not actually well-written. Dataset has been fixed.
- **Next iteration**: add a worked example for complexity-as-defect to the system prompt, then re-run. Goal is 8/8 with fabricated-major count still at zero.